# 01: Data Audit

**Last updated:** 2026-11-04

## Purpose
This notebook was made to check whether the data our model learns from is trustworthy.
This notebook only reads the data and reports what it finds, it does NOT change any files.

**Scope:** This audit covers only the data we currently have:
- **ACLED** (protests, riots, violence)
- **World Bank WDI** (GDP, inflation, trade, etc.)

New datasets from the proposal (WGI, IMF debt, FAOSTAT) are not included yet. They'll be added after this audit.

## What this notebook checks

**Part A: Was the data cleaned correctly?**
Basic checks for bugs:
1. No duplicate country-year rows
2. No missing years in the middle of a country's data
3. ACLED and World Bank data line up correctly in the merge

**Part B: Is the data good enough to trust?**
1. **Missing data:** Why is North Korea ranked #1 when most of its data is missing?
2. **COVID labels:** How many "disruption" labels come from the 2020 COVID crash?
3. **Forecast gaps:** How much data is missing in the 2025 rows we forecast from?
4. **Model agreement:** Do the score model and the alert model agree with each other?


## Summary of Findings
Part A: Pipeline has no bugs, data is cleaned, merged and labeled correctly.
Concerns found (uneven coverage, small dataset) are just about the data we have, not how it was processed.

In [1]:
import pandas as pd
print("Ready!", pd.__version__)

Ready! 3.1.0


## Setup: Loading the Data

In [2]:
# Load the model's training data and its current predictions
table = pd.read_csv("../data/processed/model_table.csv")
preds = pd.read_csv("../data/processed/predictions.csv")

print("model_table:", table.shape)
print("predictions:", preds.shape)
table.head()

model_table: (257, 50)
predictions: (257, 8)


,iso3,country,year,coverage_start,protests,riots,battles,viol_civilians,explosions,strategic_dev,...,manuf_pct_gdp_lag1,fdi_pct_gdp_lag1,log_gdp_pc_lag1,unemployment_lag1,fragility_next,disruption_next,unrest_per_m,fatalities_per_m,violent_per_m,is_forecast_row
0,AFG,Afghanistan,2017,2016,133,5,8389,289,3829,305,...,4.114197,0.516606,6.257825,10.116,0.930316,0.0,3.866745,1018.074650,350.444753,False
1,AFG,Afghanistan,2018,2016,212,10,9542,325,3735,246,...,3.530422,0.274797,6.264293,11.184,0.000000,0.0,6.041961,1177.746892,370.192569,False
2,AFG,Afghanistan,2019,2016,59,3,9150,408,4042,240,...,6.160177,0.661572,6.197131,11.192,4.933712,1.0,1.637780,1094.169157,359.254980,False
3,AFG,Afghanistan,2020,2016,141,11,6104,707,2846,436,...,7.043181,0.124496,6.207790,11.187,21.655416,1.0,3.890555,800.967950,247.178202,False
4,AFG,Afghanistan,2021,2016,229,9,5906,917,2266,460,...,7.528115,0.064994,6.235953,11.710,0.000000,1.0,5.949939,1065.289028,227.222660,False


## Part A: Was the data cleaned correctly?
Basic sanity checks on `model_table`. If any of these fail, it's a bug in the pipeline.

### A1. No duplicate rows
Each country should appear only **once per year**.

In [3]:
# Count rows where the same country + year shows up more than once
dupes = table.duplicated(subset=["iso3", "year"]).sum()
print("Duplicate country-year rows:", dupes)

Duplicate country-year rows: 0


 ### A2. No missing years
Each country's years should be continuous, with no gaps between its first and last year.

In [4]:
# For each country: first year, last year, and how many rows it has
years = table.groupby("iso3")["year"].agg(["min", "max", "count"])

# If there are no gaps, count should equal (max - min + 1)
years["expected"] = years["max"] - years["min"] + 1
years["missing_years"] = years["expected"] - years["count"]

years.sort_values("min")

,min,max,count,expected,missing_years
iso3,,,,,
VNM,2010,2025,16,16,0
THA,2010,2025,16,16,0
BGD,2010,2025,16,16,0
PAK,2010,2025,16,16,0
KHM,2010,2025,16,16,0
NPL,2010,2025,16,16,0
LKA,2010,2025,16,16,0
MMR,2010,2025,16,16,0
LAO,2011,2025,15,15,0


**Result:** No gaps, every country's years are continuous.

**Note:** Countries start in very different years (2010 to 2021) because ACLED began covering regions at different times. Late starters (AUS, NZL, PNG, TLS) have only 5 years of data, so the model knows much less about them.


### A3. Merge alignment
`model_table` should have the same country-years as the ACLED file, and World Bank values should match the original World Bank file.

In [5]:
# Load the two files that were merged together
acled = pd.read_csv("../data/processed/acled_country_year.csv")
wdi = pd.read_csv("../data/processed/wdi_country_year.csv")

# Check 1: same country-year pairs as ACLED?
acled_keys = set(zip(acled["iso3"], acled["year"]))
table_keys = set(zip(table["iso3"], table["year"]))
print("Same country-years as ACLED:", acled_keys == table_keys)

# Check 2: do World Bank values match the original file?
check = table[["iso3", "year", "gdp_growth"]].merge(
    wdi[["iso3", "year", "gdp_growth"]],
    on=["iso3", "year"],
    suffixes=("_table", "_wdi"),
)
both_blank = check["gdp_growth_table"].isna() & check["gdp_growth_wdi"].isna()
same_value = check["gdp_growth_table"] == check["gdp_growth_wdi"]
print("GDP growth mismatches:", (~(same_value | both_blank)).sum())

Same country-years as ACLED: True
GDP growth mismatches: 0


### A4. Labels are computed correctly (extra)
`disruption_next` = 1 if next year's GDP growth falls more than 3 points below the 3-year average, OR next year's inflation is above 10%.
We recompute this rule from the World Bank data and compare.

In [6]:
# Recompute the disruption label ourselves from the World Bank data
w = wdi.sort_values(["iso3", "year"]).copy()
by_country = w.groupby("iso3")

w["growth_3y_avg"] = by_country["gdp_growth"].transform(lambda s: s.rolling(3, min_periods=2).mean())
w["next_growth"] = by_country["gdp_growth"].shift(-1)
w["next_inflation"] = by_country["inflation"].shift(-1)

big_drop = (w["growth_3y_avg"] - w["next_growth"]) > 3
high_inflation = w["next_inflation"] > 10
w["my_label"] = (big_drop | high_inflation).astype(float)

# If we don't know anything about next year, there's no label
no_next_year = w["next_growth"].isna() & w["next_inflation"].isna()
w.loc[no_next_year, "my_label"] = None

# Compare with the label in model_table
check = table[["iso3", "year", "disruption_next"]].merge(w[["iso3", "year", "my_label"]], on=["iso3", "year"])
both_blank = check["disruption_next"].isna() & check["my_label"].isna()
same = check["disruption_next"] == check["my_label"]
print("Label mismatches:", (~(same | both_blank)).sum())
print("Labelled rows:", check["disruption_next"].notna().sum())
print("Disruptions (1s):", int(check["disruption_next"].sum()))


Label mismatches: 0
Labelled rows: 225
Disruptions (1s): 46


**Result:** 0 label mismatches. The labels match the rule in `pipeline/merge.py`.

**Note:** Only **225 of 257** rows have a label, and only **46** of those are disruptions (~20%).
- 24 unlabelled rows are the 2025 forecast rows (expected, since next year hasn't happened yet)
- 7 are North Korea, which has almost no World Bank data
- 1 is Afghanistan 2024

225 rows is a very small amount of data for training a model.
